In [ ]:
# In[1]:

import pandas as pd

# Load files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (rule 10)
for df in (metric_df, trace_df, log_df, error_logs_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# -- Prepare metric summaries and aggregation --
metric_sample = metric_df.head(5)

metric_summary = pd.DataFrame({
    'file': ['metric.csv'],
    'unique_cmdb_id_count': [metric_df['cmdb_id'].nunique()],
    'unique_kpi_or_name_count': [metric_df['kpi_name'].nunique()]
})

# Aggregate globally (do NOT filter by any window) and compute p95, p50, count
metric_agg = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         p50='median')
    .rename(columns={'kpi_name': 'name'})
)
metric_agg['source'] = 'metric'
metric_agg = metric_agg[['source', 'cmdb_id', 'name', 'p95', 'p50', 'count']].sort_values('count', ascending=False).head(50)

# Round numeric columns for compact display
metric_agg[['p95','p50']] = metric_agg[['p95','p50']].round(6)

# -- Prepare trace summaries and aggregation --
trace_sample = trace_df.head(5)

trace_summary = pd.DataFrame({
    'file': ['trace.csv'],
    'unique_cmdb_id_count': [trace_df['cmdb_id'].nunique()],
    'unique_kpi_or_name_count': [trace_df['trace_name'].nunique()]
})

trace_agg = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         p50='median')
    .rename(columns={'trace_name': 'name'})
)
trace_agg['source'] = 'trace'
trace_agg = trace_agg[['source', 'cmdb_id', 'name', 'p95', 'p50', 'count']].sort_values('count', ascending=False).head(50)
trace_agg[['p95','p50']] = trace_agg[['p95','p50']].round(6)

# -- Prepare log summaries and aggregation --
log_sample = log_df.head(5)

log_summary = pd.DataFrame({
    'file': ['log.csv'],
    'unique_cmdb_id_count': [log_df['cmdb_id'].nunique()],
    'unique_kpi_or_name_count': [log_df['log_name'].nunique()]
})

log_agg = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         p50='median')
    .rename(columns={'log_name': 'name'})
)
log_agg['source'] = 'log'
log_agg = log_agg[['source', 'cmdb_id', 'name', 'p95', 'p50', 'count']].sort_values('count', ascending=False).head(50)
log_agg[['p95','p50']] = log_agg[['p95','p50']].round(6)

# -- Prepare error_logs summary (no aggregation step requested for this file) --
error_sample = error_logs_df.head(5)

# For error_logs, treat 'message' as the name field for uniqueness summary
error_summary = pd.DataFrame({
    'file': ['error_logs.csv'],
    'unique_cmdb_id_count': [error_logs_df['cmdb_id'].nunique()],
    'unique_kpi_or_name_count': [error_logs_df['message'].nunique() if 'message' in error_logs_df.columns else 0]
})

# Final compact outputs (multiple variables displayed)
metric_sample, metric_summary, metric_agg, trace_sample, trace_summary, trace_agg, log_sample, log_summary, log_agg, error_sample, error_summary

```
Out[1]:
```
summary = (
    "Summary of telemetry files (based on the execution results):\n\n"
    "metric.csv\n"
    "- Sample (first 5 rows) shown for adservice: cpu, diskio, latency-50, latency-90, mem.\n"
    "- Unique cmdb_id count: 12. Unique KPI (kpi_name) count: 8.\n"
    "- Aggregated (source, cmdb_id, name) top rows mostly have count=25.\n"
    "  Examples: adservice cpu (p95≈0.709603, p50≈0.659584, count=25); "
    "adservice diskio (p95≈90039.6, p50≈66098.7, count=25); adservice mem (p95≈1.1345e8, p50≈1.1334e8, count=25).\n\n"
    "trace.csv\n"
    "- Sample (first 5 rows) shown for currencyservice trace entries.\n"
    "- Unique cmdb_id count: 8. Unique trace_name count: 44.\n"
    "- Aggregated top rows mostly have count=25. Many trace.row_count entries are large.\n"
    "  Example: frontendservice trace.to_frontendservice.row_count (p95≈8182.6, p50≈7812.0, count=25).\n\n"
    "log.csv\n"
    "- Sample (first 5 rows) shown for adservice and cartservice log.error_count and log.row_count.\n"
    "- Unique cmdb_id count: 10. Unique log_name count: 2 (log.error_count, log.row_count).\n"
    "- Aggregation shows log.error_count ≈ 0 for services; log.row_count shows request volumes.\n"
    "  Examples: adservice log.row_count (p95≈470.8, p50≈448.0, count=25); emailservice and some services have 24 or fewer points (redis had 5 points).\n\n"
    "error_logs.csv\n"
    "- Empty (no rows). Unique cmdb_id count: 0. Unique message count: 0.\n\n"
    "Overall: metric, trace, and log files contain regularly sampled data with aggregated summaries available (top 50 rows shown). error_logs.csv is empty so no raw error messages are present."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp    cmdb_id    kpi_name         value
0 2024-01-19 01:18:00+00:00  adservice         cpu  5.883311e-01
1 2024-01-19 01:18:00+00:00  adservice      diskio  8.890554e+04
2 2024-01-19 01:18:00+00:00  adservice  latency-50  3.013174e-03
3 2024-01-19 01:18:00+00:00  adservice  latency-90  4.637781e-03
4 2024-01-19 01:18:00+00:00  adservice         mem  1.133240e+08,          file  unique_cmdb_id_count  unique_kpi_or_name_count
0  metric.csv                    12                         8,     source         cmdb_id        name           p95           p50  count
0   metric       adservice         cpu  7.096030e-01  6.595840e-01     25
1   metric       adservice      diskio  9.003959e+04  6.609871e+04     25
2   metric       adservice  latency-50  3.013000e-03  2.995000e-03     25
3   metric       adservice  latency-90  4.636000e-03  4.618000e-03     25
4   metric       adservice         mem  1.134506e+08  1.133418e+08     25
..     ...             ...         ...           ...           ...    ...
45  metric  paymentservice  latency-50  5.925000e-03  4.481000e-03     25
46  metric  paymentservice  latency-90  1.272400e-02  8.985000e-03     25
47  metric  paymentservice         mem  4.062874e+07  4.041455e+07     25
48  metric  paymentservice      socket  4.000000e+00  4.000000e+00     25
49  metric  paymentservice    workload  1.089577e+00  9.356000e-01     25

[50 rows x 6 columns],                   timestamp          cmdb_id                                trace_name       value
0 2024-01-19 01:18:00+00:00  currencyservice  trace.from_frontendservice.duration_mean    0.000133
1 2024-01-19 01:18:00+00:00  currencyservice   trace.from_frontendservice.duration_p95    0.000190
2 2024-01-19 01:18:00+00:00  currencyservice     trace.from_frontendservice.error_rate    0.000000
3 2024-01-19 01:18:00+00:00  currencyservice      trace.from_frontendservice.row_count  177.000000
4 2024-01-19 01:18:00+00:00     emailservice             trace.from_root.duration_mean    0.000053,         file  unique_cmdb_id_count  unique_kpi_or_name_count
0  trace.csv                     8                        44,     source                cmdb_id                                      name          p95          p50  count
68   trace        frontendservice    trace.to_frontendservice.duration_mean     0.010899     0.007769     25
69   trace        frontendservice     trace.to_frontendservice.duration_p95     0.021673     0.014926     25
70   trace        frontendservice       trace.to_frontendservice.error_rate     0.000000     0.000000     25
71   trace        frontendservice        trace.to_frontendservice.row_count  8182.600000  7812.000000     25
75   trace        frontendservice  trace.to_productcatalogservice.row_count  3805.000000  3635.000000     25
..     ...                    ...                                       ...          ...          ...    ...
102  trace  productcatalogservice                trace.from_root.error_rate     0.000000     0.000000     25
101  trace  productcatalogservice              trace.from_root.duration_p95     0.014063     0.011105     25
100  trace  productcatalogservice             trace.from_root.duration_mean     0.005428     0.004282     25
64   trace        frontendservice    trace.to_currencyservice.duration_mean     0.000356     0.000296     25
65   trace        frontendservice     trace.to_currencyservice.duration_p95     0.000226     0.000215     25

[50 rows x 6 columns],                   timestamp          cmdb_id         log_name  value
0 2024-01-19 01:18:00+00:00        adservice  log.error_count      0
1 2024-01-19 01:18:00+00:00        adservice    log.row_count     30
2 2024-01-19 01:18:00+00:00      cartservice  log.error_count      0
3 2024-01-19 01:18:00+00:00      cartservice    log.row_count     53
4 2024-01-19 01:18:00+00:00  currencyservice  log.error_count      0,       file  unique_cmdb_id_count  unique_kpi_or_name_count
0  log.csv                    10                         2,    source                cmdb_id             name      p95     p50  count
0     log              adservice  log.error_count     0.00     0.0     25
1     log              adservice    log.row_count   470.80   448.0     25
2     log            cartservice  log.error_count     0.00     0.0     25
3     log            cartservice    log.row_count   810.40   771.0     25
7     log        currencyservice    log.row_count  2498.00  2383.0     25
6     log        currencyservice  log.error_count     0.00     0.0     25
10    log               frontend  log.error_count     0.00     0.0     25
11    log               frontend    log.row_count  2807.20  2658.0     25
19    log        shippingservice    log.row_count   499.20   448.0     25
18    log        shippingservice  log.error_count     0.00     0.0     25
14    log  recommendationservice  log.error_count     0.00     0.0     25
15    log  recommendationservice    log.row_count   583.80   558.0     25
8     log           emailservice  log.error_count     0.00     0.0     24
9     log           emailservice    log.row_count    34.85    27.5     24
5     log        checkoutservice    log.row_count   105.40    82.5     24
4     log        checkoutservice  log.error_count     0.00     0.0     24
12    log         paymentservice  log.error_count     0.00     0.0     24
13    log         paymentservice    log.row_count    69.70    55.0     24
17    log                  redis    log.row_count     5.00     5.0      5
16    log                  redis  log.error_count     0.00     0.0      5, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [],              file  unique_cmdb_id_count  unique_kpi_or_name_count
0  error_logs.csv                     0                         0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes (assumed loaded in the kernel): metric_df, trace_df, log_df
# Ensure timestamps are parsed as UTC datetimes (idempotent if already done)
for df in (metric_df, trace_df, log_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Compute global thresholds (FULL FILE, before any window filtering)
# Metrics: global P95 per (cmdb_id, kpi_name)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'kpi_name': 'name', 'value': 'p95'})
)

# Traces: global P95 and P5 per (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .rename(columns={'trace_name': 'name'})
)

# Logs: global P95 and P5 per (cmdb_id, log_name)
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .rename(columns={'log_name': 'name'})
)

# 2) Filter rows to the incident window (inclusive)
start = pd.Timestamp("2024-01-19 01:15:56", tz='UTC')
end = pd.Timestamp("2024-01-19 01:45:56", tz='UTC')

metric_win = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
trace_win = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
log_win = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# Helper to format earliest anomaly timestamp consistently
def _fmt_ts(s):
    if pd.isna(s):
        return pd.NaT
    # keep timezone-aware UTC string
    return pd.to_datetime(s).tz_convert('UTC')

# 3) Metrics anomalies: value >= global P95
if not metric_win.empty:
    m_merged = metric_win.merge(metric_thresholds, left_on=['cmdb_id', 'kpi_name'], right_on=['cmdb_id', 'name'], how='left')
    m_merged['is_anom'] = (m_merged['value'] >= m_merged['p95'])
    # Group and aggregate
    m_grp = (
        m_merged
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            window_count=('value', 'count'),
            anomalies_count=('is_anom', 'sum'),
            earliest_anomaly_timestamp=('timestamp', lambda x: x[m_merged.loc[x.index, 'is_anom']].min() if any(m_merged.loc[x.index, 'is_anom']) else pd.NaT),
            max_value_in_window=('value', 'max')
        )
    )
    # Attach global p95
    m_grp = m_grp.merge(metric_thresholds.rename(columns={'name':'kpi_name'}), on=['cmdb_id','kpi_name'], how='left')
    # Filter only series with anomalies_count>0
    m_anoms = m_grp[m_grp['anomalies_count']>0].copy()
    if not m_anoms.empty:
        m_anoms['severity_ratio'] = m_anoms['max_value_in_window'] / m_anoms['p95'].replace({0:np.nan})
        # Format timestamp column as UTC tz-aware (keep as Timestamp)
        m_anoms['earliest_anomaly_timestamp'] = m_anoms['earliest_anomaly_timestamp'].apply(_fmt_ts)
        # Select and rename columns as requested
        metric_anoms = (
            m_anoms
            .assign(source='metric')
            .rename(columns={'kpi_name':'name'})
            [['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','max_value_in_window','p95','severity_ratio']]
            .sort_values(by=['anomalies_count','severity_ratio'], ascending=[False, False])
            .head(20)
        )
        # Round numeric columns for compactness
        metric_anoms['p95'] = metric_anoms['p95'].round(6)
        metric_anoms['max_value_in_window'] = metric_anoms['max_value_in_window'].round(6)
        metric_anoms['severity_ratio'] = metric_anoms['severity_ratio'].round(6)
    else:
        metric_anoms = pd.DataFrame(columns=['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','max_value_in_window','p95','severity_ratio'])
else:
    metric_anoms = pd.DataFrame(columns=['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','max_value_in_window','p95','severity_ratio'])

# 4) Trace anomalies:
if not trace_win.empty:
    t_merged = trace_win.merge(trace_thresholds, left_on=['cmdb_id','trace_name'], right_on=['cmdb_id','name'], how='left')
    # classify row_count vs others
    t_merged['is_row_count'] = t_merged['trace_name'].str.contains('row_count', case=False, na=False)
    # anomaly condition
    t_merged['is_anom'] = np.where(t_merged['is_row_count'], t_merged['value'] <= t_merged['p5'], t_merged['value'] >= t_merged['p95'])
    # Group and aggregate
    t_grp = (
        t_merged
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            window_count=('value','count'),
            anomalies_count=('is_anom','sum'),
            earliest_anomaly_timestamp=('timestamp', lambda x: x[t_merged.loc[x.index, 'is_anom']].min() if any(t_merged.loc[x.index, 'is_anom']) else pd.NaT),
            min_value_in_window=('value','min'),
            max_value_in_window=('value','max')
        )
    )
    # attach p95/p5
    t_grp = t_grp.merge(trace_thresholds.rename(columns={'name':'trace_name'}), on=['cmdb_id','trace_name'], how='left')
    t_anoms = t_grp[t_grp['anomalies_count']>0].copy()
    if not t_anoms.empty:
        t_anoms['earliest_anomaly_timestamp'] = t_anoms['earliest_anomaly_timestamp'].apply(_fmt_ts)
        t_anoms = (
            t_anoms
            .assign(source='trace')
            .rename(columns={'trace_name':'name'})
            [['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','min_value_in_window','max_value_in_window','p95','p5']]
            .sort_values(by=['anomalies_count'], ascending=False)
            .head(20)
        )
        t_anoms['p95'] = t_anoms['p95'].round(6)
        t_anoms['p5'] = t_anoms['p5'].round(6)
        t_anoms['min_value_in_window'] = t_anoms['min_value_in_window'].round(6)
        t_anoms['max_value_in_window'] = t_anoms['max_value_in_window'].round(6)
        trace_anoms = t_anoms
    else:
        trace_anoms = pd.DataFrame(columns=['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','min_value_in_window','max_value_in_window','p95','p5'])
else:
    trace_anoms = pd.DataFrame(columns=['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','min_value_in_window','max_value_in_window','p95','p5'])

# 5) Log anomalies:
if not log_win.empty:
    l_merged = log_win.merge(log_thresholds, left_on=['cmdb_id','log_name'], right_on=['cmdb_id','name'], how='left')
    # classify: row_count drop (<=p5), error_count spike (>=p95)
    l_merged['is_row_count'] = l_merged['log_name'].str.contains('row_count', case=False, na=False)
    l_merged['is_error_count'] = l_merged['log_name'].str.contains('error_count', case=False, na=False)
    l_merged['is_anom'] = np.where(l_merged['is_row_count'], l_merged['value'] <= l_merged['p5'],
                                   np.where(l_merged['is_error_count'], l_merged['value'] >= l_merged['p95'], False))
    # Group and aggregate
    l_grp = (
        l_merged
        .groupby(['cmdb_id','log_name'], as_index=False)
        .agg(
            window_count=('value','count'),
            anomalies_count=('is_anom','sum'),
            earliest_anomaly_timestamp=('timestamp', lambda x: x[l_merged.loc[x.index, 'is_anom']].min() if any(l_merged.loc[x.index, 'is_anom']) else pd.NaT),
            min_value_in_window=('value','min'),
            max_value_in_window=('value','max')
        )
    )
    # attach thresholds
    l_grp = l_grp.merge(log_thresholds.rename(columns={'name':'log_name'}), on=['cmdb_id','log_name'], how='left')
    l_anoms = l_grp[l_grp['anomalies_count']>0].copy()
    if not l_anoms.empty:
        l_anoms['earliest_anomaly_timestamp'] = l_anoms['earliest_anomaly_timestamp'].apply(_fmt_ts)
        l_anoms = (
            l_anoms
            .assign(source='log')
            .rename(columns={'log_name':'name'})
            [['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','min_value_in_window','max_value_in_window','p95','p5']]
            .sort_values(by=['anomalies_count'], ascending=False)
            .head(20)
        )
        l_anoms['p95'] = l_anoms['p95'].round(6)
        l_anoms['p5'] = l_anoms['p5'].round(6)
        l_anoms['min_value_in_window'] = l_anoms['min_value_in_window'].round(6)
        l_anoms['max_value_in_window'] = l_anoms['max_value_in_window'].round(6)
        log_anoms = l_anoms
    else:
        log_anoms = pd.DataFrame(columns=['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','min_value_in_window','max_value_in_window','p95','p5'])
else:
    log_anoms = pd.DataFrame(columns=['source','cmdb_id','name','window_count','anomalies_count','earliest_anomaly_timestamp','min_value_in_window','max_value_in_window','p95','p5'])

# Final compact outputs: metric_anoms, trace_anoms, log_anoms
metric_anoms, trace_anoms, log_anoms

```
Out[2]:
```
summary = (
    "Summary of anomalies in the incident window (2024-01-19 01:15:56 → 01:45:56 UTC):\n\n"
    "Metrics:\n"
    "- Widespread 'socket' series hit their global P95 across many services (adservice, cartservice, currencyservice, emailservice, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice, frontend, checkoutservice). Earliest socket hits appear at 2024-01-19 01:18:00 UTC. Most socket series show max == p95 (severity_ratio ≈ 1.0).\n"
    "- The most notable, high-severity metric is checkoutservice.diskio: a spike in-window with max ≈ 681,686.6 vs global p95 ≈ 123,539.6 (severity_ratio ≈ 5.52), earliest anomaly 2024-01-19 01:18:00 UTC — this is the clearest actionable spike.\n"
    "- Many 'diskio' and 'error' metric anomalies are flagged because their global p95 equals 0; those should be treated cautiously as likely threshold/artifact-driven detections.\n\n"
    "Traces:\n"
    "- Many trace 'error_rate' series were flagged because global p95/p5 are 0 and values are 0 (so value >= p95). These do not indicate non-zero errors — they are artifacts of zero thresholds. No broad duration or traffic (row_count) anomalies with meaningful non-zero thresholds were detected in this summary.\n\n"
    "Logs:\n"
    "- log.error_count is flagged across many services but global p95 = 0 and values = 0, so these are not actionable errors (artifact of zero threshold).\n"
    "- Several services show brief drops in log.row_count (treated as traffic drops) with anomalies_count = 2 in-window for services such as adservice, cartservice, emailservice, checkoutservice, currencyservice, frontend, paymentservice, recommendationservice, shippingservice. Earliest row_count drops are around 01:18:00 or 01:27:00 UTC depending on service. Redis has very small sample size (5 points).\n\n"
    "Overall conclusions and next steps:\n"
    "- Investigate checkoutservice.diskio spike first (high severity_ratio ≈ 5.5) — check disk I/O, logs, and workload on checkoutservice around 01:18 UTC.\n"
    "- Review socket-related behavior across services (many series at their P95) to determine whether they represent a platform-wide socket saturation or if thresholds are too low.\n"
    "- Treat many flagged zero-threshold anomalies (error_count, diskio with p95=0, trace error_rate) as threshold artifacts; adjust thresholds or inspect raw series before acting.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    source                cmdb_id    name  window_count  anomalies_count earliest_anomaly_timestamp  max_value_in_window            p95  severity_ratio
5   metric              adservice  socket            25               25  2024-01-19 01:18:00+00:00             4.000000       4.000000        1.000000
12  metric            cartservice  socket            25               25  2024-01-19 01:18:00+00:00             5.000000       5.000000        1.000000
26  metric        currencyservice  socket            25               25  2024-01-19 01:18:00+00:00             4.000000       4.000000        1.000000
33  metric           emailservice  socket            25               25  2024-01-19 01:18:00+00:00             3.000000       3.000000        1.000000
48  metric         paymentservice  socket            25               25  2024-01-19 01:18:00+00:00             4.000000       4.000000        1.000000
56  metric  productcatalogservice  socket            25               25  2024-01-19 01:18:00+00:00             4.000000       4.000000        1.000000
63  metric  recommendationservice  socket            25               25  2024-01-19 01:18:00+00:00             4.000000       4.000000        1.000000
68  metric                  redis  socket            25               25  2024-01-19 01:18:00+00:00             4.000000       4.000000        1.000000
73  metric        shippingservice  socket            25               25  2024-01-19 01:18:00+00:00             2.000000       2.000000        1.000000
8   metric            cartservice   error            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
22  metric        currencyservice  diskio            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
29  metric           emailservice  diskio            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
36  metric               frontend   error            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
42  metric      frontend-external   error            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
51  metric  productcatalogservice  diskio            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
52  metric  productcatalogservice   error            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
59  metric  recommendationservice  diskio            25               25  2024-01-19 01:18:00+00:00             0.000000       0.000000             NaN
40  metric               frontend  socket            25               17  2024-01-19 01:18:00+00:00            24.000000      24.000000        1.000000
19  metric        checkoutservice  socket            25               11  2024-01-19 01:32:00+00:00            12.000000      12.000000        1.000000
15  metric        checkoutservice  diskio            25                2  2024-01-19 01:18:00+00:00        681686.595659  123539.640775        5.517958,     source                cmdb_id                                         name  window_count  anomalies_count earliest_anomaly_timestamp  min_value_in_window  max_value_in_window  p95   p5
58   trace        frontendservice                   trace.from_root.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
66   trace        frontendservice          trace.to_currencyservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
70   trace        frontendservice          trace.to_frontendservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
74   trace        frontendservice    trace.to_productcatalogservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
138  trace                   root          trace.to_frontendservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
146  trace                   root    trace.to_productcatalogservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
134  trace                   root             trace.to_emailservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
150  trace                   root    trace.to_recommendationservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
94   trace  productcatalogservice        trace.from_frontendservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
98   trace  productcatalogservice  trace.from_recommendationservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
102  trace  productcatalogservice                   trace.from_root.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
106  trace  recommendationservice        trace.from_frontendservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
114  trace  recommendationservice                   trace.from_root.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
118  trace  recommendationservice    trace.to_productcatalogservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
110  trace  recommendationservice  trace.from_recommendationservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
122  trace  recommendationservice    trace.to_recommendationservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
78   trace        frontendservice    trace.to_recommendationservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
50   trace           emailservice                   trace.from_root.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
54   trace        frontendservice        trace.from_frontendservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0
38   trace        currencyservice        trace.from_frontendservice.error_rate            25               25  2024-01-19 01:18:00+00:00                  0.0                  0.0  0.0  0.0,    source                cmdb_id             name  window_count  anomalies_count earliest_anomaly_timestamp  min_value_in_window  max_value_in_window      p95       p5
0     log              adservice  log.error_count            25               25  2024-01-19 01:18:00+00:00                    0                    0     0.00     0.00
2     log            cartservice  log.error_count            25               25  2024-01-19 01:18:00+00:00                    0                    0     0.00     0.00
10    log               frontend  log.error_count            25               25  2024-01-19 01:18:00+00:00                    0                    0     0.00     0.00
6     log        currencyservice  log.error_count            25               25  2024-01-19 01:18:00+00:00                    0                    0     0.00     0.00
18    log        shippingservice  log.error_count            25               25  2024-01-19 01:18:00+00:00                    0                    0     0.00     0.00
14    log  recommendationservice  log.error_count            25               25  2024-01-19 01:18:00+00:00                    0                    0     0.00     0.00
8     log           emailservice  log.error_count            24               24  2024-01-19 01:19:00+00:00                    0                    0     0.00     0.00
12    log         paymentservice  log.error_count            24               24  2024-01-19 01:19:00+00:00                    0                    0     0.00     0.00
4     log        checkoutservice  log.error_count            24               24  2024-01-19 01:19:00+00:00                    0                    0     0.00     0.00
16    log                  redis  log.error_count             5                5  2024-01-19 01:19:00+00:00                    0                    0     0.00     0.00
17    log                  redis    log.row_count             5                5  2024-01-19 01:19:00+00:00                    5                    5     5.00     5.00
1     log              adservice    log.row_count            25                2  2024-01-19 01:18:00+00:00                   30                  475   470.80   405.80
3     log            cartservice    log.row_count            25                2  2024-01-19 01:18:00+00:00                   53                  822   810.40   711.20
9     log           emailservice    log.row_count            24                2  2024-01-19 01:27:00+00:00                   20                   36    34.85    21.15
5     log        checkoutservice    log.row_count            24                2  2024-01-19 01:27:00+00:00                   60                  109   105.40    63.45
7     log        currencyservice    log.row_count            25                2  2024-01-19 01:18:00+00:00                  177                 2618  2498.00  2208.20
11    log               frontend    log.row_count            25                2  2024-01-19 01:18:00+00:00                  194                 2813  2807.20  2466.40
13    log         paymentservice    log.row_count            24                2  2024-01-19 01:27:00+00:00                   40                   72    69.70    42.30
15    log  recommendationservice    log.row_count            25                2  2024-01-19 01:18:00+00:00                   42                  595   583.80   513.60
19    log        shippingservice    log.row_count            25                2  2024-01-19 01:18:00+00:00                   34                  516   499.20   380.40)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing dataframes/thresholds from the kernel state
# Ensure timestamp parsed (idempotent)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# Incident window
start = pd.Timestamp("2024-01-19 01:15:56", tz='UTC')
end = pd.Timestamp("2024-01-19 01:45:56", tz='UTC')

# Ensure metric_thresholds exists and has columns ['cmdb_id','name','p95']
# (it was computed earlier in the session). If kpi name column is 'name', we'll merge accordingly.
# Filter metric_df to candidates and window
metric_focus_win = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

# Merge global p95 thresholds (calculated on full file earlier)
# metric_thresholds has columns ['cmdb_id','name','p95']
m = metric_focus_win.merge(metric_thresholds, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','name'], how='left')

# Determine anomaly boolean per point using global p95 (value >= p95)
m['is_anom'] = (m['value'] >= m['p95'])

# Helper to compute longest consecutive anomaly run (in minutes) for a group's sorted timestamps
def longest_consecutive_run(ts_series, is_anom_series):
    # ts_series: pd.Series of timestamps (tz-aware)
    # is_anom_series: boolean series aligned
    if not is_anom_series.any():
        return 0
    # take timestamps where anomaly true
    anom_ts = ts_series[is_anom_series].sort_values()
    if anom_ts.empty:
        return 0
    # convert to integer seconds
    seconds = anom_ts.view('int64') // 10**9  # numpy int representation
    # compute differences between consecutive anomaly timestamps
    diffs = np.diff(seconds)
    # runs: count consecutive diffs == 60 (one-minute cadence). Start a new run when diff != 60
    runs = []
    run_len = 1
    for d in diffs:
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    # run lengths are in number of points (minutes)
    return int(max(runs)) if runs else 1

# Group by series and compute required aggregates
agg_rows = []
grouped = m.groupby(['cmdb_id','kpi_name'], as_index=False)

for (cmdb_id, kpi_name), g in grouped:
    window_count = int(g.shape[0])
    anomalies_count = int(g['is_anom'].sum())
    if anomalies_count == 0:
        continue
    earliest_anom_ts = g.loc[g['is_anom'], 'timestamp'].min() if anomalies_count>0 else pd.NaT
    max_value_in_window = float(g['value'].max())
    # global p95 from merged column (should be same for group)
    global_p95 = float(g['p95'].iloc[0]) if pd.notna(g['p95'].iloc[0]) else np.nan
    severity_ratio = (max_value_in_window / global_p95) if (pd.notna(global_p95) and global_p95!=0) else np.nan
    # longest consecutive run
    longest_run = longest_consecutive_run(g['timestamp'], g['is_anom'])
    agg_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'window_count': window_count,
        'anomalies_count': anomalies_count,
        'earliest_anomaly_timestamp': pd.to_datetime(earliest_anom_ts).tz_convert('UTC') if pd.notna(earliest_anom_ts) else pd.NaT,
        'longest_consecutive_anomaly_run_min': longest_run,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95,
        'severity_ratio': severity_ratio
    })

metric_focus_df = pd.DataFrame(agg_rows)

# Sort and limit: by severity_ratio desc (NaNs last), then anomalies_count desc
if not metric_focus_df.empty:
    metric_focus_df['severity_ratio_sort'] = metric_focus_df['severity_ratio'].replace({np.nan:-np.inf})
    metric_focus_df = metric_focus_df.sort_values(by=['severity_ratio_sort','anomalies_count'], ascending=[False, False]).drop(columns=['severity_ratio_sort'])
    # Round numeric columns for compactness
    metric_focus_df['max_value_in_window'] = metric_focus_df['max_value_in_window'].round(6)
    metric_focus_df['global_p95'] = metric_focus_df['global_p95'].round(6)
    metric_focus_df['severity_ratio'] = metric_focus_df['severity_ratio'].round(6)
    metric_focus_df = metric_focus_df[['cmdb_id','kpi_name','window_count','anomalies_count','earliest_anomaly_timestamp','longest_consecutive_anomaly_run_min','max_value_in_window','global_p95','severity_ratio']].head(50)
else:
    # empty table with requested columns
    metric_focus_df = pd.DataFrame(columns=['cmdb_id','kpi_name','window_count','anomalies_count','earliest_anomaly_timestamp','longest_consecutive_anomaly_run_min','max_value_in_window','global_p95','severity_ratio'])

# 4) Also return available KPI names per candidate component (unique kpi_name even if no anomalies)
kpi_names = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby('cmdb_id', as_index=False)['kpi_name']
    .unique()
)
# Convert numpy arrays to comma-joined strings for compact display
kpi_names['kpi_names'] = kpi_names['kpi_name'].apply(lambda arr: ','.join(map(str, arr)))
kpi_list_df = kpi_names[['cmdb_id','kpi_names']]

# Display compact results: metric_focus_df and kpi_list_df
metric_focus_df, kpi_list_df

```
Out[3]:
```
Summary of focused metric analysis for the five candidate components (window 2024-01-19 01:15:56 → 01:45:56 UTC):

Key findings
- Checkoutservice: highest-priority anomaly is checkoutservice.diskio — a brief spike with max ≈ 681,686.6 vs global P95 ≈ 123,539.6 (severity_ratio ≈ 5.52). Earliest anomaly 2024-01-19 01:18:00 UTC. This is a clear, high-severity disk I/O spike and should be investigated first.
- Emailservice: detected anomalies in workload (severity_ratio ≈ 1.25) and latency metrics (latency-90 and latency-50 with severity_ratio ≈ 1.20 and 1.07). Earliest anomalies ≈ 01:32–01:33 UTC — investigate emailservice latency and workload around those times.
- Checkoutservice again shows a smaller workload anomaly (severity_ratio ≈ 1.24) at ~01:33 UTC.
- Many series for these components (diskio and error) were flagged with anomalies_count = 25 but have global P95 == 0, producing NaN severity_ratio. Those are likely threshold artifacts (zero P95) and require manual inspection of raw series or threshold tuning rather than immediate action.

Other notes
- Longest consecutive anomaly runs are mostly short (1 minute) for the non-artifact anomalies listed above. Several zero-threshold series show long runs (25 minutes) because every in-window point meets the zero threshold.
- Overall, 36 series with anomalies were detected across the five components; the most actionable is checkoutservice.diskio (large spike), followed by emailservice workload/latency and checkoutservice workload.

Available KPI names per component (compact)
- checkoutservice: cpu, diskio, latency-50, latency-90, mem, socket, workload, error
- currencyservice: cpu, diskio, latency-50, latency-90, mem, socket, workload, error
- emailservice: cpu, diskio, latency-50, latency-90, mem, socket, workload, error
- productcatalogservice: cpu, diskio, error, latency-50, latency-90, mem, socket, workload
- recommendationservice: cpu, diskio, latency-50, latency-90, mem, socket, workload

Recommended next steps
1. Immediately investigate checkoutservice.diskio (disk I/O metrics, logs, processes, storage subsystem) around 01:18 UTC.
2. Inspect emailservice and checkoutservice request/latency traces and logs around 01:32–01:33 UTC for root causes of workload/latency increases.
3. Treat series with p95 == 0 as artifacts; review global thresholds and raw time series before acting on those.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  window_count  anomalies_count earliest_anomaly_timestamp  longest_consecutive_anomaly_run_min  max_value_in_window     global_p95  severity_ratio
1         checkoutservice      diskio            25                2  2024-01-19 01:18:00+00:00                                    1        681686.595659  123539.640775        5.517958
20           emailservice    workload            25                2  2024-01-19 01:33:00+00:00                                    1             1.359917       1.089553        1.248141
6         checkoutservice    workload            25                2  2024-01-19 01:33:00+00:00                                    1             1.353417       1.093970        1.237161
17           emailservice  latency-90            25                2  2024-01-19 01:32:00+00:00                                    1             0.007422       0.006172        1.202478
16           emailservice  latency-50            25                2  2024-01-19 01:32:00+00:00                                    1             0.003531       0.003287        1.074386
..                    ...         ...           ...              ...                        ...                                  ...                  ...            ...             ...
8         currencyservice      diskio            25               25  2024-01-19 01:18:00+00:00                                   25             0.000000       0.000000             NaN
15           emailservice      diskio            25               25  2024-01-19 01:18:00+00:00                                   25             0.000000       0.000000             NaN
22  productcatalogservice      diskio            25               25  2024-01-19 01:18:00+00:00                                   25             0.000000       0.000000             NaN
23  productcatalogservice       error            25               25  2024-01-19 01:18:00+00:00                                   25             0.000000       0.000000             NaN
30  recommendationservice      diskio            25               25  2024-01-19 01:18:00+00:00                                   25             0.000000       0.000000             NaN

[36 rows x 9 columns],                  cmdb_id                                          kpi_names
0        checkoutservice  cpu,diskio,latency-50,latency-90,mem,socket,wo...
1        currencyservice  cpu,diskio,latency-50,latency-90,mem,socket,wo...
2           emailservice  cpu,diskio,latency-50,latency-90,mem,socket,wo...
3  productcatalogservice  cpu,diskio,error,latency-50,latency-90,mem,soc...
4  recommendationservice  cpu,diskio,latency-50,latency-90,mem,socket,wo...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```